# Module 5: Answer Agent with Self-Critique

This notebook verifies the AnswerAgent, illustrating parallel text generation for predicted questions, critique verification of numbers and facts, and answer revisions based on critique feedback.

In [ ]:
# Cell 1: Initialization
import sys
from loguru import logger

logger.remove()
logger.add(sys.stderr, level="INFO")

from extraction_agent import EmbeddingService
from predictive_analyst_agent import PredictiveAnalystAgent
from answer_agent import AnswerAgent

embedding_service = EmbeddingService()
analyst_agent = PredictiveAnalystAgent(embedding_service)
answer_agent = AnswerAgent()
print("Agents initialized successfully!")

In [ ]:
# Cell 2: Fetch and Analyze Questions for Reliance Industries
print("Fetching question list...")
results = await analyst_agent.run("Reliance Industries")
answerable_questions = results["answerable"]
print(f"Found {len(answerable_questions)} answerable questions to process.")

In [ ]:
# Cell 3: Process Q&A in Batches (Top 10)
print("Running batch answering and self-critique...")
qa_pairs = await answer_agent.run_batch(answerable_questions[:10])
print("Batch run complete.")

In [ ]:
# Cell 4: View QA Pairs and Critique Status
print("=== TOP 10 Q&A PAIRS WITH CRITIQUE STATUS ===")
for idx, qa in enumerate(qa_pairs):
    status = "PASS" if qa.critique_passed else "FAIL"
    print(f"\n#{idx+1} Question: {qa.question}")
    print(f"Topic: {qa.topic} | Adversarial Score: {qa.adversarial_score}")
    print(f"Critique Status: {status} | Confidence: {qa.confidence}")
    if qa.critique_issues:
        print(f"Issues Flagged: {qa.critique_issues}")
    print(f"Answer: {qa.final_answer}")

In [ ]:
# Cell 5: Show Example of Revised Answer
revised_examples = [qa for qa in qa_pairs if qa.revised_answer]
if not revised_examples:
    print("No answers required revision during this run (all original answers passed the critique fact check).")
else:
    example = revised_examples[0]
    print("=== EXAMPLE OF ANSWER REVISION ===")
    print(f"Question: {example.question}")
    print(f"Original Answer: {example.answer}")
    print(f"Critique Issues: {example.critique_issues}")
    print(f"Revised Answer: {example.revised_answer}")